In [1]:
from google.colab import drive
drive.mount('/content/drive')

import torch
try:
    major_version, minor_version = torch.cuda.get_device_capability()
    !pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

    if major_version >= 8:
        !pip install --no-deps xformers trl peft accelerate bitsandbytes
    else:
        !pip install --no-deps xformers trl peft accelerate bitsandbytes

    print("Cài đặt xong")
except Exception as e:
    print(f"Lỗi: Hãy đảm bảo bạn đã bật GPU Chi tiết: {e}")

Mounted at /content/drive
  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-6rcc2ndx/unsloth_32bec63c99c94d1aa9db02d6af5005cf
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-6rcc2ndx/unsloth_32bec63c99c94d1aa9db02d6af5005cf
  Resolved https://github.com/unslothai/unsloth.git to commit e0e606a24a96d8053dbce3adbf9bf71ce2d2f70a
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 45.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 122.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 428.0/428.0 kB 37.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 125.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185

In [5]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from unsloth import FastLanguageModel
import torch

mBERT_PATH = "/content/drive/MyDrive/MLLA/mBERT_FakeNews_Model"
QWEN_PATH = "/content/drive/MyDrive/MLLA/Qwen_FakeNews_Model"

print("⏳ Đang nạp mô hình từ bộ nhớ Colab...")

mbert_tokenizer = AutoTokenizer.from_pretrained(mBERT_PATH)
mbert_model = AutoModelForSequenceClassification.from_pretrained(mBERT_PATH).to("cuda")

qwen_model, qwen_tokenizer = FastLanguageModel.from_pretrained(
    model_name = QWEN_PATH,
    max_seq_length = 1024,
    load_in_4bit = True,
)
FastLanguageModel.for_inference(qwen_model)

⏳ Đang nạp mô hình từ bộ nhớ Colab...


Loading weights:   0%|          | 0/201 [00:01<?, ?it/s]

==((====))==  Unsloth 2026.5.2: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/1.53G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

...

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

unsloth/qwen2.5-1.5b-instruct-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Unsloth 2026.5.2 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(151936, 1536, padding_idx=151665)
        (layers): ModuleList(
          (0): Qwen2DecoderLayer(
            (self_attn): Qwen2Attention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=1536, out_features=1536, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=1536, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=1536, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.Linear(
       

In [8]:
import gradio as gr
import re
import torch


# Hàm suy luận cho mBERT
def predict_mbert(text):
    # mBERT giới hạn 512 token
    inputs = mbert_tokenizer(text, return_tensors="pt", truncation=True, max_length=512, padding=True).to("cuda")
    with torch.no_grad():
        outputs = mbert_model(**inputs)
        prediction = torch.argmax(outputs.logits, dim=1).item()
    return "🚨 TIN GIẢ (Fake News)" if prediction == 1 else "✅ TIN THẬT (Real News)"

# Hàm suy luận cho Qwen2.5-1.5B
def predict_qwen(text):
    # Dùng System Prompt kỷ luật như lúc bạn train
    messages = [
        {"role": "system", "content": "Bạn là chuyên gia kiểm chứng tin tức. Hãy phân tích bài báo và chỉ in ra số 0 (Tin thật) hoặc 1 (Tin giả)."},
        {"role": "user", "content": text}
    ]
    inputs = qwen_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = qwen_tokenizer(inputs, return_tensors="pt").to("cuda")

    with torch.no_grad():
        # Ép mô hình chỉ sinh tối đa 5 token để lấy số 0 hoặc 1
        outputs = qwen_model.generate(**inputs, max_new_tokens=5, pad_token_id=qwen_tokenizer.eos_token_id)

    response = qwen_tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

    # Dùng biểu thức chính quy (Regex) để bắt lấy số 0 hoặc 1
    match = re.search(r'[01]', response)
    if match:
        return "🚨 TIN GIẢ (Fake News)" if match.group() == '1' else "✅ TIN THẬT (Real News)"
    return "⚠️ Lỗi: Không thể xác định nhãn từ Qwen."

# Hàm gộp kết quả đưa lên giao diện
def analyze_news(text, selected_mode):
    if not text.strip():
        return "Vui lòng nhập nội dung bài báo!"

    if "mBERT" in selected_mode:
        res = predict_mbert(text)
        return f"Kết quả từ mBERT:\n\n{res}"

    elif "Qwen" in selected_mode:
        res = predict_qwen(text)
        return f"Kết quả từ Qwen2.5-1.5B:\n\n{res}"

    else: # Chế độ So sánh cả 2
        res_mbert = predict_mbert(text)
        res_qwen = predict_qwen(text)
        return f"⚖️ KẾT QUẢ ĐỐI SÁNH:\n\n- mBERT chẩn đoán:\t{res_mbert}\n- Qwen chẩn đoán:\t{res_qwen}"


custom_css = """
.model-card { border: 1px solid var(--border-color-primary); border-radius: 8px; padding: 16px; background-color: var(--background-fill-secondary); height: 100%;}
.model-card h3 { margin-top: 0; color: var(--body-text-color); font-size: 1.1em; font-weight: 600;}
.model-card p { color: var(--body-text-color-subdued); font-size: 0.9em; margin-bottom: 15px; line-height: 1.5;}
.tag { background-color: var(--background-fill-primary); color: var(--body-text-color); padding: 4px 10px; border-radius: 12px; font-size: 0.75em; font-weight: 500; margin-right: 6px; border: 1px solid var(--border-color-primary); display: inline-block;}
"""

with gr.Blocks(css=custom_css, theme=gr.themes.Default(neutral_hue="slate")) as demo:
    with gr.Row():
        gr.HTML("""
            <div class="model-card">
                <h3>⚙️ Qwen2.5-1.5B</h3>
                <p>LLM lớn, fine-tuned LoRA. Khả năng đọc ngữ cảnh dài 1024 tokens.</p>
                <div><span class="tag">LoRA</span><span class="tag">Unsloth</span><span class="tag">4-bit</span></div>
            </div>
        """)
        gr.HTML("""
            <div class="model-card">
                <h3>🔤 mBERT</h3>
                <p>Mô hình Baseline ngôn ngữ song hướng. Tốc độ nhận diện cực nhanh.</p>
                <div><span class="tag">bert-multilingual</span><span class="tag">512-tokens</span></div>
            </div>
        """)

    mode = gr.Radio(
        choices=["⚙️ Qwen2.5", "🔤 mBERT", "🔄 So sánh (Cả hai mô hình)"],
        value="🔄 So sánh (Cả hai mô hình)",
        label="Chọn mô hình kiểm chứng"
    )

    with gr.Group():
        input_text = gr.Textbox(show_label=False, placeholder="Dán hoặc gõ nội dung bài báo cần xác minh tại đây...", lines=7)

    submit_btn = gr.Button("🔍 Phân tích bài báo", variant="primary", size="lg")
    output_result = gr.Textbox(label="Kết quả phân tích", interactive=False, lines=5)

    submit_btn.click(fn=analyze_news, inputs=[input_text, mode], outputs=output_result)

# Khởi chạy ứng dụng Web và tạo link chia sẻ
demo.launch(share=True, debug=True)

/tmp/ipykernel_4443/2687051732.py:63: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(css=custom_css, theme=gr.themes.Default(neutral_hue="slate")) as demo:
/tmp/ipykernel_4443/2687051732.py:63: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(css=custom_css, theme=gr.themes.Default(neutral_hue="slate")) as demo:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://56988a8b1042e8f7f0.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://56988a8b1042e8f7f0.gradio.live
